### `26_xgboost_per_series_paper_taxonomy` — только **XGBoost** (per-series) + таксономия признаков (features)

Идеи статьи Cao & Li (2020) «Research on the DouDiZhu's playing strategy based on XGBoost», *International Journal of Wireless and Mobile Computing* ([Inderscience](https://www.inderscience.com/info/inarticle.php?artid=112556), DOI [10.1504/IJWMC.2020.112556](https://doi.org/10.1504/IJWMC.2020.112556)) переносятся на прогноз **числа операций** (`fact_value`) так:

| Идея статьи | Смысл для прогноза | В этом ноутбуке |
|-------------|-------------------|-----------------|
| Много подзадач (multi-classification / legal moves) | Декомпозиция по контексту | Отдельный **XGBRegressor** на каждый `operation_id` (per-series), аналог *character training* |
| Представление состояния (state representation) | Признаки без утечки (leakage) в будущее | Лаги (lags) `fact_value` + при варианте `+C-exog` — календарь (calendar), **`calculate_minute_lag1`** (минуты за месяц **M−1** при таргете месяц **M**) и **только лаги** `calc_*_lagN` из `CALC_WHITELIST` |
| Подбор параметров XGBoost | Устойчивость и обобщение | Несколько пресетов: `XGB-default`, `XGB-regularized`, и те же с суффиксом `+C-exog` |
| Rule correction | Ограничения домена | После предсказания: `max(0, ·)`; в метриках как в ноутбуке 25 |

**Рекурсивный горизонт (recursive forecast)** с экзогенными колонками: на шаге прогноза месяца **M** в признаки входят календарь **самого M** (`month`, `year`, `quarter`, `workDayMonth_2`), **`calculate_minute_lag1`** (минуты расчёта за **M−1**, доступны к моменту прогноза M) и лаги `calc_*`. Лаги `fact_value` на тесте обновляются **прогнозами** предыдущих шагов (авторегрессия (autoregression)). Сырые `calculate_minute` в **X не подаются**.

См. также описание наборов A/B/C в [`memory/14_notebook_features.md`](../memory/14_notebook_features.md).

---

#### Таксономия признаков (по [`memory/ml_features_description.md`](../memory/ml_features_description.md))

| Группа | Роль (аналог статьи) | Примеры колонок | В текущем конвейере |
|--------|---------------------|-----------------|---------------------|
| **CalendarSeasonality** | Сезонность календаря | `month`, `year`, `quarter`, `workDayMonth_2` | `quarter` добавляется в коде; `workDayMonth_2` из flatten `calculate_parameter` |
| **StateDynamics** | Динамика ряда | лаги `y_{t-1}..y_{t-L}` | В XGB (режим без exog): авто-лаги из истории `fact_value`; в `df` также `lag_1`, `lag_12`, `rolling_mean_3` |
| **RuleHistory** | История «правил расчёта» | `calculate_minute_lag1` (= M−1), `calc_*_lag1..lag12` (whitelist C) | Варианты `+C-exog`; исходное поле `calculate_minute` только для сдвига |
| **BusinessCapacity** | Ёмкость / нормативы | `staff_count`, `capacity_hours`, `norm_value`, … | В сырых CSV этого ноутбука **нет** — зарезервировано для расширения |
| **HierarchyContext** | Контекст подразделения | `total_resist_fact_lag_1`, `share_in_resist_lag_1` | Не строится в per-series-only пайплайне |
| **StaticSeriesId** | Идентификатор ряда | `operation_id` | Разбиение `groupby("operation_id")`; глобальный `op_code` здесь не в признаках X |

В коде ниже словарь **`FEATURE_TAXONOMY`** печатается для каждого `resist_id` после feature engineering.


In [ ]:

import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None
    print("Установите пакет: pip install xgboost")

_ROOT = Path("..").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
from scripts.calc_lag_features import add_calc_shift_lags, calc_lag_columns_for_sets

%matplotlib inline
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message="divide by zero encountered in log",
    category=RuntimeWarning,
)


#### Конфигурация

In [ ]:

RESIST_ID = 124
RESIST_IDS = [124]
SUMMARY_TOP_PER_RID = 5
EXCEL_ITOG_TOP = 20

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]
TEST_YEAR = 2025
TEST_MONTHS = [7, 8, 9, 10, 11, 12]

CALC_WHITELIST = [
    "calc_workDayMonth_2",
    "calc_hourInWeek_12",
    "calc_numberWorkHour_1",
    "calc_effective_13",
    "calc_absenteeism_14",
    "calc_employFact_7",
    "calc_resist_10",
]

LOCAL_MAX_LAG = 6
OUT_XLSX = Path("../data/processed/26_xgboost_per_series_paper_taxonomy.xlsx")


#### Загрузка данных и `df_model39`

In [ ]:
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 139,
]
PREDICT_COMMENT = "first_predict_value_record"


def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def flatten_calc_to_dict(calc_obj: dict, prefix: str = "calc_") -> dict:
    out = {}
    for k, v in (calc_obj or {}).items():
        col = f"{prefix}{k}"
        if isinstance(v, dict):
            if "value" in v:
                out[col] = v.get("value")
            else:
                out[col] = json.dumps(v, ensure_ascii=False) if v else ""
        else:
            out[col] = v
    return out


dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError(f"Файлы данных не найдены в {RAW_DIR}")

df_all = pd.concat(dfs, ignore_index=True)
df_all["period"] = pd.to_datetime(df_all["period"], errors="coerce")
df_all["resist_id"] = pd.to_numeric(df_all["resist_id"], errors="coerce").astype("Int64")

bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])
df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

mask_ok = (
    ~df_all["cancel_approve"]
    & ~mask_predict_record
    & ~df_all["resist_id"].isin(EXCLUDED_RESIST_IDS)
)
df_ok = df_all[mask_ok].copy()

records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    calc_flat = flatten_calc_to_dict(calc_obj)
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        if to_bool_value(payload.get("operation_active", payload.get("active"))) is False:
            continue
        base_record = {
            "resist_id": row.get("resist_id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": calc_flat.get("calc_workDayMonth_2"),
            "calculate_minute": payload.get("calculate_minute"),
        }
        records.append({**base_record, **calc_flat})


def coerce_numeric_series(series: pd.Series) -> pd.Series:
    """pd.to_numeric (to_numeric) + повтор для строк с десятичной запятой (decimal comma), напр. \"8,5\"."""
    out = pd.to_numeric(series, errors="coerce")
    mask = out.isna() & series.notna()
    if mask.any():
        fixed = (
            series[mask]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
        )
        out.loc[mask] = pd.to_numeric(fixed, errors="coerce")
    return out


df_model39 = pd.DataFrame(records)
df_model39["fact_value"] = coerce_numeric_series(df_model39["fact_value"])
df_model39["fact_predict"] = coerce_numeric_series(df_model39["fact_predict"])
df_model39["workDayMonth_2"] = coerce_numeric_series(df_model39["workDayMonth_2"])
df_model39["calculate_minute"] = coerce_numeric_series(df_model39["calculate_minute"])

calc_cols = [c for c in df_model39.columns if c.startswith("calc_")]
for col in calc_cols:
    df_model39[col] = coerce_numeric_series(df_model39[col])
print(f"calc_* колонки ({len(calc_cols)}):", calc_cols)

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1), errors="coerce"
).dt.date
df_model39 = df_model39.dropna(subset=["fact_value"])
df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

RESIST_IDS = list(dict.fromkeys(RESIST_IDS))
avail = set(int(x) for x in df_model39["resist_id"].dropna().unique())
missing = [r for r in RESIST_IDS if r not in avail]
if missing:
    sample = sorted(avail)[:15]
    raise ValueError(
        f"Нет данных для resist_id={missing}. Примеры доступных id: {sample}..."
    )
print(f"RESIST_IDS ({len(RESIST_IDS)}): {RESIST_IDS}")


#### Метрики

In [ ]:

def calc_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[float, float, float]:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(np.asarray(y_pred, dtype=float), 0)
    y_pred = np.where(y_pred < 1e-3, 0.0, y_pred)
    if len(y_true) == 0:
        return 0.0, 0.0, 0.0
    acc = np.empty_like(y_true)
    mask_zero = y_true == 0
    acc[mask_zero] = np.where(y_pred[mask_zero] == 0, 100.0, 0.0)
    mask_ok = ~mask_zero
    if np.any(mask_ok):
        acc[mask_ok] = np.clip(
            100.0 - np.abs(y_true[mask_ok] - y_pred[mask_ok]) / np.abs(y_true[mask_ok]) * 100.0,
            0, 100,
        )
    tochnost = round(float(np.mean(acc)), 2)
    stab = 0.0
    if tochnost > 0:
        cv = np.std(acc, ddof=0) / tochnost
        stab = round(max(0.0, (1 - cv)) * 100, 2)
    kachestvo = 0.0
    if tochnost + stab > 0:
        kachestvo = round(2 * tochnost * stab / (tochnost + stab), 2)
    return tochnost, stab, kachestvo


#### Обучение: только XGBoost per-series (`add_calc_shift_lags`, whitelist C)

In [ ]:

from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=PerformanceWarning)


def _statsmodels_module_tag(warning) -> str:
    return ""


def interpret_warning(msg: str, category) -> str:
    cname = getattr(category, "__name__", str(category))
    mod = getattr(category, "__module__", "") or ""
    s = str(msg).lower()
    if "xgboost" in mod.lower():
        return "XGBoost: см. сообщение (данные, параметры дерева или DMatrix)."
    if "sklearn" in mod.lower():
        return "Sklearn: не сходимость итераций или плохо обусловленные признаки."
    if "runtime" in cname.lower():
        return "RuntimeWarning: деление на ноль, переполнение или NaN в вычислениях."
    return (
        "См. текст сообщения; типично: короткий или нестабильный ряд, экстремальные значения, "
        "численная неустойчивость."
    )


def append_warning_rows(
    rows: list,
    *,
    resist_id,
    operation_id,
    model_kind: str,
    model_label: str,
    phase: str,
    n_train_points,
    caught,
):
    for w in caught:
        msg = str(w.message)
        cat = w.category
        if issubclass(cat, PerformanceWarning):
            continue
        interp = interpret_warning(msg, cat)
        rows.append(
            {
                "resist_id": resist_id,
                "operation_id": operation_id,
                "model_kind": model_kind,
                "model_label": model_label,
                "phase": phase,
                "warning_category": getattr(cat, "__name__", str(cat)),
                "message": msg,
                "interpretation": interp,
                "n_train_points": n_train_points if model_kind == "per_series" else "",
                "statsmodels_module": _statsmodels_module_tag(w),
            }
        )


def _local_effective_max_lag(n_train: int, max_lag: int) -> int:
    if n_train < 3:
        return 1
    return max(1, min(max_lag, n_train - 2))


def _local_build_xy(y_arr: np.ndarray, max_lag: int):
    y_arr = np.asarray(y_arr, dtype=float)
    X, y_t = [], []
    for t in range(max_lag, len(y_arr)):
        X.append([y_arr[t - j] for j in range(1, max_lag + 1)])
        y_t.append(y_arr[t])
    return np.array(X, dtype=float), np.array(y_t, dtype=float)


def _local_build_xy_exog(y_arr: np.ndarray, max_lag: int, exog_arr: np.ndarray):
    # exog_arr[i] aligned with y_arr[i], same length as y_arr
    y_arr = np.asarray(y_arr, dtype=float)
    X, y_t = [], []
    for t in range(max_lag, len(y_arr)):
        lag = [y_arr[t - j] for j in range(1, max_lag + 1)]
        X.append(lag + list(exog_arr[t]))
        y_t.append(y_arr[t])
    return np.array(X, dtype=float), np.array(y_t, dtype=float)


def _local_recursive_forecast_xgb(model, y_hist, n_fc: int, L: int, exog_test_arr=None):
    history = list(np.asarray(y_hist, dtype=float))
    fc = []
    fallback = float(np.mean(history)) if history else 0.0
    use_exog = exog_test_arr is not None and exog_test_arr.size > 0 and exog_test_arr.shape[1] > 0
    for h in range(n_fc):
        if len(history) < L:
            pred = fallback
        else:
            lag_part = [history[-j] for j in range(1, L + 1)]
            if use_exog:
                row = np.array([lag_part + list(exog_test_arr[h])], dtype=float)
            else:
                row = np.array([lag_part], dtype=float)
            pred = float(model.predict(row)[0])
        fc.append(pred)
        history.append(pred)
    return np.array(fc, dtype=float)


def train_local_xgb_per_series(
    df_full,
    mask_train,
    mask_test,
    resist_id,
    series_config_name,
    warning_rows,
    *,
    exog_cols: list[str],
    xgb_kw: dict,
    max_lag=None,
):
    if XGBRegressor is None:
        raise RuntimeError("xgboost не установлен")
    if max_lag is None:
        max_lag = LOCAL_MAX_LAG
    base_kw = dict(verbosity=0, random_state=42, n_jobs=-1)
    base_kw.update(xgb_kw)
    test_idx = df_full.loc[mask_test].index
    preds = pd.Series(0.0, index=test_idx)
    n_exog = len(exog_cols)
    for op_id, grp in df_full.groupby("operation_id"):
        train_part = grp[mask_train.reindex(grp.index, fill_value=False)].sort_values("date")
        test_part = grp[mask_test.reindex(grp.index, fill_value=False)].sort_values("date")
        if test_part.empty:
            continue
        y_hist = train_part["fact_value"].values.astype(float)
        n_fc = len(test_part)
        n_tr = len(y_hist)
        L = _local_effective_max_lag(n_tr, max_lag)
        try:
            if n_exog > 0:
                ex_tr = train_part[exog_cols].to_numpy(dtype=float, copy=True)
                ex_tr = np.nan_to_num(ex_tr, nan=0.0, posinf=0.0, neginf=0.0)
                X, y_tgt = _local_build_xy_exog(y_hist, L, ex_tr)
            else:
                X, y_tgt = _local_build_xy(y_hist, L)
            if X.shape[0] < 2:
                raise ValueError("too few supervised rows")
            kw = dict(base_kw)
            ne = int(kw.get("n_estimators", 400))
            kw["n_estimators"] = min(ne, max(50, n_tr * 25))
            model = XGBRegressor(**kw)
            with warnings.catch_warnings(record=True) as w_fit:
                warnings.simplefilter("always")
                model.fit(X, y_tgt)
            append_warning_rows(
                warning_rows,
                resist_id=resist_id,
                operation_id=op_id,
                model_kind="per_series",
                model_label=series_config_name,
                phase="fit",
                n_train_points=n_tr,
                caught=w_fit,
            )
            if n_exog > 0:
                ex_te = test_part[exog_cols].to_numpy(dtype=float, copy=True)
                ex_te = np.nan_to_num(ex_te, nan=0.0, posinf=0.0, neginf=0.0)
                fc = _local_recursive_forecast_xgb(model, y_hist, n_fc, L, ex_te)
            else:
                fc = _local_recursive_forecast_xgb(model, y_hist, n_fc, L, None)
        except Exception:
            fc = np.full(n_fc, np.mean(y_hist) if len(y_hist) > 0 else 0)
        preds.loc[test_part.index] = np.maximum(fc, 0)
    return preds.values


XGB_KW_DEFAULT = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
)
XGB_KW_REG = dict(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    reg_alpha=0.1,
    reg_lambda=5,
    subsample=0.85,
    colsample_bytree=0.85,
)

SERIES_CONFIGS = {
    "XGB-default": {"use_exog": False, "xgb_kw": XGB_KW_DEFAULT},
    "XGB-regularized": {"use_exog": False, "xgb_kw": XGB_KW_REG},
    "XGB-default+C-exog": {"use_exog": True, "xgb_kw": XGB_KW_DEFAULT},
    "XGB-regularized+C-exog": {"use_exog": True, "xgb_kw": XGB_KW_REG},
}

all_results: list = []
all_predictions: dict = {}
df_by_rid: dict = {}
mask_test_by_rid: dict = {}
summary_parts: list[str] = []
warning_rows: list[dict] = []

for rid in RESIST_IDS:
    df = df_model39[df_model39["resist_id"] == rid].copy()
    df["quarter"] = ((df["month"].astype(int) - 1) // 3 + 1).astype(int)

    group = df.groupby("operation_id")
    df["lag_1"] = group["fact_value"].shift(1)
    df["lag_12"] = group["fact_value"].shift(12)
    df["rolling_mean_3"] = group["fact_value"].transform(
        lambda s: s.shift(1).rolling(window=3, min_periods=1).mean()
    )
    # calculate_minute доступны на месяц M−1 относительно таргета M → в X только лаг (lag) 1 по ряду
    df["calculate_minute_lag1"] = group["calculate_minute"].shift(1)
    num_cols = df.select_dtypes(include=["number"]).columns
    df[num_cols] = df[num_cols].fillna(0)
    df = df.infer_objects(copy=False)
    if df.isnull().any().any():
        df = df.fillna(0)

    df, _lag_calc_cols, calc_keys_used = add_calc_shift_lags(df, drop_scalar_calc=True)
    _b_flat, c_flat = calc_lag_columns_for_sets(calc_keys_used, whitelist=CALC_WHITELIST)

    calendar_cols = ["month", "year", "quarter", "workDayMonth_2"]
    exog_cols_full = [c for c in calendar_cols + ["calculate_minute_lag1"] + c_flat if c in df.columns]

    FEATURE_TAXONOMY = {
        "CalendarSeasonality": [c for c in calendar_cols if c in df.columns],
        "StateDynamics": [f"autoregressive_lag_1..{LOCAL_MAX_LAG} (в матрице XGB)"],
        "RuleHistory": [c for c in ["calculate_minute_lag1"] + c_flat if c in df.columns],
        "BusinessCapacity": [],
        "HierarchyContext": [],
        "StaticSeriesId": ["operation_id (через groupby, не колонка в X)"],
    }
    print(f"resist_id={rid} FEATURE_TAXONOMY:\n{json.dumps(FEATURE_TAXONOMY, ensure_ascii=False, indent=2)}")

    mask_test = (df["year"] == TEST_YEAR) & (df["month"].isin(TEST_MONTHS))
    mask_train = ~mask_test
    y_test = df.loc[mask_test, "fact_value"]

    df_by_rid[rid] = df
    mask_test_by_rid[rid] = mask_test

    results_rid: list = []
    preds_rid: dict = {}

    n_test = int(mask_test.sum())
    t_fp, s_fp, q_fp = calc_metrics(y_test.values, df.loc[mask_test, "fact_predict"].values)

    if XGBRegressor is None:
        print("Пропуск обучения: нет xgboost")
        break

    if mask_train.sum() == 0:
        y_zero = np.zeros(n_test)
        t_z, s_z, q_z = calc_metrics(y_test.values, y_zero)
        for config_name in SERIES_CONFIGS:
            results_rid.append({
                "resist_id": rid,
                "Конфигурация": config_name,
                "Набор признаков": "—",
                "Тип": "per-series",
                "Точность, %": t_z,
                "Стабильность, %": s_z,
                "Качество, %": q_z,
                "Время, сек": 0.0,
                "Точн. fact_predict, %": t_fp,
                "Стаб. fact_predict, %": s_fp,
                "Кач. fact_predict, %": q_fp,
            })
            preds_rid[config_name] = y_zero
        all_results.extend(results_rid)
        all_predictions[rid] = preds_rid
        summary_parts.append(f"{rid} baseline:{q_fp:.0f}%  forecast:{max(r['Качество, %'] for r in results_rid):.0f}%")
        print(", ".join(summary_parts))
        continue

    for config_name, cfg in SERIES_CONFIGS.items():
        ex_cols = exog_cols_full if cfg["use_exog"] else []
        t0 = time.perf_counter()
        y_pred = train_local_xgb_per_series(
            df,
            mask_train,
            mask_test,
            rid,
            config_name,
            warning_rows,
            exog_cols=ex_cols,
            xgb_kw=cfg["xgb_kw"],
        )
        elapsed = round(time.perf_counter() - t0, 2)
        tochnost, stab, kachestvo = calc_metrics(y_test.values, y_pred)
        results_rid.append({
            "resist_id": rid,
            "Конфигурация": config_name,
            "Набор признаков": "—",
            "Тип": "per-series",
            "Точность, %": tochnost,
            "Стабильность, %": stab,
            "Качество, %": kachestvo,
            "Время, сек": elapsed,
            "Точн. fact_predict, %": t_fp,
            "Стаб. fact_predict, %": s_fp,
            "Кач. fact_predict, %": q_fp,
        })
        preds_rid[config_name] = y_pred

    all_results.extend(results_rid)
    all_predictions[rid] = preds_rid
    q_best = max(r["Качество, %"] for r in results_rid)
    summary_parts.append(f"{rid} baseline:{q_fp:.0f}%  forecast:{q_best:.0f}%")
    print(", ".join(summary_parts))

results = all_results


#### Сводка результатов

In [ ]:
df_results = pd.DataFrame(results)
df_results["Комбинация"] = df_results.apply(
    lambda r: f"{r['Конфигурация']} | {r['Набор признаков']}" if r["Набор признаков"] != "—" else r["Конфигурация"],
    axis=1,
)
df_results_full = df_results.sort_values(
    ["resist_id", "Качество, %"], ascending=[True, False]
).reset_index(drop=True)
df_results = (
    df_results_full.groupby("resist_id", group_keys=False)
    .head(SUMMARY_TOP_PER_RID)
    .reset_index(drop=True)
)
print(f"Строк в полной сводке: {len(df_results_full)}; в таблице ниже — топ-{SUMMARY_TOP_PER_RID} на каждый resist_id")
df_results


In [ ]:
TOP_N = SUMMARY_TOP_PER_RID
for rid in RESIST_IDS:
    sub = df_results_full[df_results_full["resist_id"] == rid].head(TOP_N)
    if sub.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, max(6, TOP_N * 0.45)))
    metrics_to_plot = ["Точность, %", "Стабильность, %", "Качество, %"]
    for ax, metric in zip(axes, metrics_to_plot):
        df_plot = sub.sort_values(metric, ascending=True)
        colors = ["#FF9800"] * len(df_plot)
        ax.barh(df_plot["Комбинация"], df_plot[metric], color=colors)
        ax.set_xlabel(metric)
        ax.set_title(metric)
    plt.suptitle(f"TOP-{TOP_N} комбинаций (resist_id={rid})", fontsize=14)
    plt.tight_layout()
    plt.show()

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid]
    print(f"\n--- resist_id={rid}: лучший набор по каждой модели ---")
    for cfg_name in df_r["Конфигурация"].unique():
        best_row = df_r[df_r["Конфигурация"] == cfg_name].iloc[0]
        print(f"  {cfg_name:<22} → {best_row['Набор признаков']:<7} | Качество={best_row['Качество, %']:.2f}%")


#### Диагностика лучшей комбинации по операциям

In [ ]:
per_rid_best = {}

for rid in RESIST_IDS:
    df_r = df_results_full[df_results_full["resist_id"] == rid].sort_values(
        "Качество, %", ascending=False
    )
    if df_r.empty:
        continue
    best_row = df_r.iloc[0]
    best_combo = best_row["Комбинация"]
    best_config = best_row["Конфигурация"]
    best_fs = best_row["Набор признаков"]
    best_type = best_row["Тип"]
    print(f"\n{'='*60}\nresist_id={rid}\nЛучшая комбинация по Качеству: {best_combo} ({best_type})")
    print(f"  Модель: {best_config}, Набор: {best_fs}")
    print(
        f"  Точность={best_row['Точность, %']}% | Стабильность={best_row['Стабильность, %']}% | "
        f"Качество={best_row['Качество, %']}%"
    )

    best_pred = all_predictions[rid][best_combo]
    df = df_by_rid[rid]
    mask_test = mask_test_by_rid[rid]
    df_test = df.loc[mask_test].copy()
    df_test["predict_new"] = best_pred

    rows_op = []
    for (r_id, op_id), grp in df_test.groupby(["resist_id", "operation_id"]):
        t1, s1, q1 = calc_metrics(grp["fact_value"].values, grp["fact_predict"].values)
        t2, s2, q2 = calc_metrics(grp["fact_value"].values, grp["predict_new"].values)
        rows_op.append({
            "resist_id": r_id,
            "operation_id": op_id,
            "Точность (fact_predict)": t1,
            "Стабильность (fact_predict)": s1,
            "Качество (fact_predict)": q1,
            f"Точность ({best_combo})": t2,
            f"Стабильность ({best_combo})": s2,
            f"Качество ({best_combo})": q2,
        })

    df_metrics_op = pd.DataFrame(rows_op)
    df_detail = df_test[["resist_id", "operation_id", "year", "month",
                         "fact_value", "fact_predict", "predict_new"]].copy()
    df_detail = df_detail.sort_values(["resist_id", "operation_id", "year", "month"]).reset_index(drop=True)
    df_detail = df_detail.rename(columns={
        "fact_value": "fact_value (факт)",
        "fact_predict": "fact_predict (текущий)",
        "predict_new": f"{best_combo} (новый)",
    })
    per_rid_best[rid] = {
        "best_combo": best_combo,
        "df_metrics_op": df_metrics_op,
        "df_detail": df_detail,
    }
    print(f"Операций в тесте: {len(df_metrics_op)}")

if RESIST_IDS:
    r0 = RESIST_IDS[0]
    best_combo = per_rid_best[r0]["best_combo"]
    df_metrics_op = per_rid_best[r0]["df_metrics_op"]
    print("\n(первый id в списке: пример df_metrics_op.head())")
    print(df_metrics_op.head(10))


#### Выгрузка в Excel

In [ ]:
from openpyxl import load_workbook

out_path = OUT_XLSX
out_path.parent.mkdir(parents=True, exist_ok=True)

WARN_COLS = [
    "resist_id",
    "operation_id",
    "model_kind",
    "model_label",
    "phase",
    "warning_category",
    "message",
    "interpretation",
    "n_train_points",
    "statsmodels_module",
]

sheet_triples = []
for rid in RESIST_IDS:
    df_itog = df_results_full[df_results_full["resist_id"] == rid].head(EXCEL_ITOG_TOP)
    if rid not in per_rid_best:
        continue
    b = per_rid_best[rid]
    df_warn = pd.DataFrame([r for r in warning_rows if r["resist_id"] == rid])
    if df_warn.empty:
        df_warn = pd.DataFrame(columns=WARN_COLS)
    else:
        df_warn = df_warn.reindex(columns=WARN_COLS)
    sheet_triples.append((rid, df_itog, b["df_detail"], b["df_metrics_op"], df_warn))

names_to_remove = []
for rid, _, _, _, _ in sheet_triples:
    names_to_remove.extend(
        [f"{rid} Итог", f"{rid} Данные теста", f"{rid} По операциям", f"{rid} Предупреждения"]
    )

if out_path.exists():
    wb = load_workbook(out_path)
    for sn in names_to_remove:
        if sn in wb.sheetnames:
            wb.remove(wb[sn])
    if len(wb.sheetnames) == 0:
        wb.create_sheet("_tmp")
    wb.save(out_path)
    wb.close()
    writer_mode = "a"
else:
    writer_mode = "w"

with pd.ExcelWriter(out_path, engine="openpyxl", mode=writer_mode) as w:
    for rid, df_itog, df_detail, df_metrics_op, df_warn in sheet_triples:
        df_itog.to_excel(w, sheet_name=f"{rid} Итог", index=False)
        df_detail.to_excel(w, sheet_name=f"{rid} Данные теста", index=False)
        df_metrics_op.to_excel(w, sheet_name=f"{rid} По операциям", index=False)
        df_warn.to_excel(w, sheet_name=f"{rid} Предупреждения", index=False)

wb = load_workbook(out_path)
if "_tmp" in wb.sheetnames and len(wb.sheetnames) > 1:
    wb.remove(wb["_tmp"])
    wb.save(out_path)
wb.close()

print(f"Excel: {out_path.resolve()}")
for rid, df_itog, _, _, df_warn in sheet_triples:
    print(
        f"  «{rid} Итог» ({len(df_itog)} строк), «{rid} Данные теста», «{rid} По операциям», "
        f"«{rid} Предупреждения» ({len(df_warn)} строк)"
    )


---

Конец ноутбука.